# PageRank with Sparse Matrix — 10⁶ Links

This notebook implements the **PageRank algorithm** on a randomly generated graph with **1,000,000 edges** using a **Compressed Sparse Row (CSR)** matrix for memory-efficient computation.

### Key design choices
| Choice | Detail |
|--------|--------|
| Graph size | 10,000 nodes, 1,000,000 directed edges |
| Sparse format | `scipy.sparse.csr_matrix` |
| Iteration | Power method (matrix-vector multiply) |
| Damping factor | 0.85 (standard Google value) |
| Convergence | L1 norm < 1e-6 |

In [ ]:
import numpy as np
import scipy.sparse as sp
import time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

## 1. Graph Generation

We randomly generate `N_EDGES` directed edges among `N_NODES` nodes. Duplicate edges are deduplicated so the graph is a proper directed graph.

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
N_NODES  = 10_000      # number of pages
N_EDGES  = 1_000_000   # number of directed links
DAMPING  = 0.85        # damping factor d
MAX_ITER = 100         # safety cap on iterations
TOL      = 1e-6        # convergence tolerance (L1 norm)
SEED     = 42

rng = np.random.default_rng(SEED)

t0 = time.perf_counter()

# Generate random edges (src → dst)
src = rng.integers(0, N_NODES, size=N_EDGES)
dst = rng.integers(0, N_NODES, size=N_EDGES)

# Remove self-loops
mask = src != dst
src, dst = src[mask], dst[mask]

# Deduplicate edges using unique on packed int64
packed = src.astype(np.int64) * N_NODES + dst.astype(np.int64)
packed = np.unique(packed)
src = (packed // N_NODES).astype(np.int32)
dst = (packed  % N_NODES).astype(np.int32)

n_edges_actual = len(src)
print(f'Nodes  : {N_NODES:,}')
print(f'Edges  : {n_edges_actual:,}  (after dedup & self-loop removal)')
print(f'Generated in {time.perf_counter()-t0:.2f}s')

## 2. Build the Sparse Transition Matrix

PageRank works on the **column-stochastic** transition matrix **M** where:

$$M_{ij} = \frac{1}{\text{out-degree}(j)} \quad \text{if edge } j \to i \text{ exists}$$

Dangling nodes (out-degree = 0) are handled by treating them as linking to all nodes uniformly.

In [ ]:
t0 = time.perf_counter()

# Out-degree of each node
out_degree = np.bincount(src, minlength=N_NODES).astype(np.float64)

# Dangling nodes: out-degree == 0
dangling_mask = (out_degree == 0)
n_dangling = dangling_mask.sum()
print(f'Dangling nodes (no outgoing links): {n_dangling:,}')

# Weight of each edge = 1 / out_degree[src]
# Avoid division by zero — dangling nodes are handled separately
safe_out = np.where(out_degree > 0, out_degree, 1.0)
data = 1.0 / safe_out[src]

# CSR: rows = dst (who receives), cols = src (who sends)
# This is the column-stochastic matrix in CSR format
M = sp.csr_matrix(
    (data, (dst, src)),
    shape=(N_NODES, N_NODES),
    dtype=np.float64
)

mem_mb = (M.data.nbytes + M.indices.nbytes + M.indptr.nbytes) / 1024**2
print(f'Sparse matrix built in {time.perf_counter()-t0:.2f}s')
print(f'CSR memory usage : {mem_mb:.1f} MB')
print(f'Density          : {n_edges_actual / N_NODES**2 * 100:.3f}%')

## 3. Power Iteration

The PageRank vector **r** is the stationary distribution of the random-surfer model:

$$\mathbf{r}^{(t+1)} = d \cdot M \mathbf{r}^{(t)} 
  + d \cdot \frac{\|\mathbf{r}_{\text{dangling}}^{(t)}\|_1}{N} \mathbf{1}
  + \frac{1-d}{N} \mathbf{1}$$

where $d$ is the damping factor and the second term redistributes probability mass from dangling nodes.

In [ ]:
# Initialise rank vector uniformly
r = np.full(N_NODES, 1.0 / N_NODES, dtype=np.float64)

teleport = (1.0 - DAMPING) / N_NODES   # base teleportation
history  = []                           # track residual per iteration

t0 = time.perf_counter()

for iteration in range(1, MAX_ITER + 1):
    # Dangling-node contribution (redistribute their mass uniformly)
    dangling_contrib = DAMPING * r[dangling_mask].sum() / N_NODES

    # Core sparse matrix-vector product
    r_new = DAMPING * M.dot(r) + dangling_contrib + teleport

    # Convergence check (L1 norm of change)
    residual = np.abs(r_new - r).sum()
    history.append(residual)

    r = r_new

    if iteration % 10 == 0 or residual < TOL:
        print(f'  iter {iteration:3d}  |  residual = {residual:.2e}')

    if residual < TOL:
        print(f'\n✓ Converged in {iteration} iterations ({time.perf_counter()-t0:.2f}s)')
        break
else:
    print(f'\n⚠ Did not converge within {MAX_ITER} iterations.')

## 4. Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(range(1, len(history)+1), history, color='steelblue', lw=2, marker='o', ms=4)
ax.axhline(TOL, color='crimson', ls='--', lw=1.5, label=f'Tolerance = {TOL}')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('L1 Residual (log scale)', fontsize=12)
ax.set_title('PageRank Power-Iteration Convergence', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Results & Statistics

In [ ]:
print('=== PageRank Summary ===')
print(f'Sum of all ranks  : {r.sum():.6f}  (should be ≈1.0)')
print(f'Min rank          : {r.min():.6e}')
print(f'Max rank          : {r.max():.6e}')
print(f'Mean rank         : {r.mean():.6e}  (= 1/N = {1/N_NODES:.6e})')
print(f'Std dev           : {r.std():.6e}')

# Top-10 nodes
top10_idx = np.argsort(r)[::-1][:10]
print('\n--- Top 10 Nodes by PageRank ---')
print(f'{"Rank":>4}  {"Node ID":>8}  {"Score":>12}  {"Out-degree":>10}')
print('-' * 40)
for rank, node in enumerate(top10_idx, 1):
    print(f'{rank:>4}  {node:>8}  {r[node]:>12.6e}  {int(out_degree[node]):>10}')

## 6. Distribution of PageRank Scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Log-scale histogram (typical for PageRank — power-law tail)
log_r = np.log10(r)
axes[0].hist(log_r, bins=60, color='steelblue', edgecolor='white', lw=0.3)
axes[0].set_xlabel('log₁₀(PageRank score)', fontsize=12)
axes[0].set_ylabel('Node count', fontsize=12)
axes[0].set_title('Distribution of PageRank Scores (log scale)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Rank vs score (log-log — should approximate power law)
sorted_r = np.sort(r)[::-1]
axes[1].loglog(np.arange(1, N_NODES+1), sorted_r, color='darkorange', lw=1.5)
axes[1].set_xlabel('Rank (log scale)', fontsize=12)
axes[1].set_ylabel('PageRank score (log scale)', fontsize=12)
axes[1].set_title('Rank vs Score (log-log)', fontsize=13, fontweight='bold')
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Memory & Performance Summary

In [ ]:
dense_mb = N_NODES**2 * 8 / 1024**2  # float64 dense matrix

print('=== Memory Comparison ===')
print(f'Dense matrix (N×N float64) : {dense_mb:>10.1f} MB')
print(f'CSR sparse matrix           : {mem_mb:>10.1f} MB')
print(f'Compression ratio           : {dense_mb/mem_mb:>10.1f}×')
print()
print('=== Complexity ===')
print(f'Space  : O(E) = O({n_edges_actual:,})')
print(f'Time   : O(k·E) where k = iterations to convergence')
print(f'        = O({len(history)} × {n_edges_actual:,}) = O({len(history)*n_edges_actual:,})')

In [ ]:

# ============================================================
# Comparing Uniform vs Random Initialization in PageRank
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

alphas = [0.50, 0.70, 0.85, 0.95]
tol = 1e-6
max_iter = 200

uniform_iters = []
random_iters = []

for alpha in alphas:

    # --------------------------------------------------------
    # Uniform initialization
    # --------------------------------------------------------
    pr_uniform = np.ones(n) / n

    for k in range(max_iter):
        new_pr = alpha * (P.T @ pr_uniform) + (1 - alpha) / n

        if np.linalg.norm(new_pr - pr_uniform, 1) < tol:
            uniform_iters.append(k + 1)
            break

        pr_uniform = new_pr

    # --------------------------------------------------------
    # Random initialization
    # --------------------------------------------------------
    rng = np.random.default_rng(42)

    pr_random = rng.random(n)
    pr_random = pr_random / pr_random.sum()

    for k in range(max_iter):
        new_pr = alpha * (P.T @ pr_random) + (1 - alpha) / n

        if np.linalg.norm(new_pr - pr_random, 1) < tol:
            random_iters.append(k + 1)
            break

        pr_random = new_pr

# ============================================================
# Plot comparison
# ============================================================

plt.figure(figsize=(7,5))

plt.plot(alphas, uniform_iters, marker='o', linewidth=2,
         label='Uniform Initialization')

plt.plot(alphas, random_iters, marker='s', linewidth=2,
         label='Random Initialization')

plt.xlabel("Damping Factor α")
plt.ylabel("Iterations to Convergence")
plt.title("Effect of Initialization on PageRank Convergence")
plt.grid(True, alpha=0.3)
plt.legend()

plt.show()

# ============================================================
# Print numerical comparison
# ============================================================

print("Alpha | Uniform Init | Random Init")
print("-" * 40)

for a, u, r in zip(alphas, uniform_iters, random_iters):
    print(f"{a:.2f}   | {u:^13} | {r:^11}")
